# Deep Learning 075 — The Geometry of Self-Attention

Companion notebook to the lesson. The usual picture is that self-attention acts like
gravity: "money" pulls "bank" toward it. The picture is right, and the exact version of
it says considerably more.

Everything below is reproduced from scratch:

| Claim | Number |
|---|---|
| The fraction of the gap closed **is** the attention weight | asserted to 1e-12 |
| The output can never leave the convex hull of the values | 0 escapes in 20,000 trials |
| Output length is set by attention peakedness alone | 0.182 measured, 0.183 predicted |
| "y_bank ends up nearer e_money" with a learned `W_v` | 68.2% — and a **random** vector scores 68.3% |
| One parameterless layer collapses a sentence | pairwise cosine 0.8994 → 1.0000 |

Only `numpy` and `scikit-learn` are needed. The one plot cell uses `matplotlib` and can
be skipped.

In [ ]:
import numpy as np

rng = np.random.default_rng(0)

def softmax(z, axis=-1):
    z = z - z.max(axis=axis, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=axis, keepdims=True)

def unit(m):
    return m / np.linalg.norm(m, axis=-1, keepdims=True)

def cos(a, b):
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))

## Part A — Where the output can land

Take the lesson's two-word sentence in 2-D, and the simplest case where `W_v` is the
identity so the **value vectors are the embeddings themselves**. Then

$$y_\text{bank} = w_1 e_\text{money} + w_2 e_\text{bank}, \qquad w_1 + w_2 = 1$$

and that is the equation of the straight segment joining the two points. The attention
weight picks the position along it — nothing else in the mechanism can move it off.

In [ ]:
e_money = np.array([2.0, 5.0])
e_bank = np.array([7.0, 3.0])
gap = np.linalg.norm(e_bank - e_money)
print(f"distance between the two embeddings: {gap:.4f}\n")

print(f"{'w on money':>12}{'y_bank':>18}{'dist to money':>16}{'gap closed':>13}")
for w1 in (0.0, 0.2, 0.5, 0.8, 1.0):
    y = w1 * e_money + (1 - w1) * e_bank
    d = np.linalg.norm(y - e_money)
    closed = 1 - d / gap
    assert abs(closed - w1) < 1e-12          # <- the claim, asserted
    print(f"{w1:>12.2f}{str(np.round(y, 2)):>18}{d:>16.4f}{closed:>12.1%}")

The assertion is the point of the cell. **The fraction of the distance closed equals the
attention weight exactly**, at every weight, to 1e-12.

So "money pulled bank toward it" has a precise strength. The lesson's example weight of
0.2 moves the word 20% of the way — a real move, and rather less than the arrows in the
usual diagram suggest.

In [ ]:
# Optional: draw it. Skip this cell if matplotlib is not installed.
import matplotlib.pyplot as plt

ws = np.linspace(0, 1, 51)
seg = np.array([w * e_money + (1 - w) * e_bank for w in ws])

fig, ax = plt.subplots(figsize=(5.5, 4.5))
ax.plot(seg[:, 0], seg[:, 1], lw=2, label="every reachable y_bank")
for p, name in ((e_bank, "e_bank"), (e_money, "e_money")):
    ax.annotate(name, p, textcoords="offset points", xytext=(6, 6))
    ax.arrow(0, 0, p[0], p[1], head_width=0.15, length_includes_head=True, alpha=.5)
y02 = 0.2 * e_money + 0.8 * e_bank
ax.scatter(*y02, s=60, zorder=3)
ax.annotate("y_bank, w=0.2", y02, textcoords="offset points", xytext=(6, -14))
ax.set(xlim=(0, 8), ylim=(0, 6), xlabel="dim 1", ylabel="dim 2",
       title="the reachable set is the segment")
ax.legend(); ax.grid(alpha=.3); plt.tight_layout(); plt.show()

## Part B — It can never leave the convex hull

Three words give a triangle, four a tetrahedron, `n` words the **convex hull** of the
`n` value vectors. Softmax weights are non-negative and sum to one, so the output is a
convex combination and lives inside that hull. It is not a tendency — it is the
definition.

A quick consequence to test: the output can never be longer than the longest value.

In [ ]:
worst, escapes = 0.0, 0
for _ in range(20_000):
    n = int(rng.integers(2, 12))
    V = rng.normal(size=(n, 8))
    w = softmax(rng.normal(size=n) * 3)
    ratio = np.linalg.norm(w @ V) / np.linalg.norm(V, axis=1).max()
    worst = max(worst, ratio)
    escapes += ratio > 1 + 1e-12

print(f"max ||y|| / max_j ||v_j|| : {worst:.6f}")
print(f"outputs outside the hull  : {escapes}")

**Zero escapes.** No choice of weights produces one, because there is no such choice.

The consequence is a hard ceiling on a single attention layer: *a representation that is
not somewhere between the meanings present in the sentence is unreachable.* "Not hot" is
not between "not" and "hot". Getting there is the job of the nonlinear feed-forward
network that follows attention inside a transformer block — attention decides what to
mix, the layer after it decides what the mixture becomes.

## Part C — And interpolating shrinks the vector

Averaging vectors that point in different directions gives something shorter than either.
For near-orthogonal unit values the cross terms cancel and

$$\|y\|^2 = \sum_i\sum_j w_i w_j (v_i \cdot v_j) \;\approx\; \sum_j w_j^2$$

so the length is set by **how peaked the attention is, and by nothing else**.

In [ ]:
print(f"{'n':>4}{'attention':>12}{'max w':>9}{'entropy':>10}{'||y||':>9}{'sqrt(sum w^2)':>16}")
for n, temp, name in [(4, 0.0, "uniform"), (4, 1.0, "mild"), (4, 4.0, "peaked"),
                      (30, 0.0, "uniform"), (30, 1.0, "mild"),
                      (30, 4.0, "peaked"), (30, 12.0, "argmax")]:
    ys, mw, ent, pred = [], [], [], []
    for _ in range(2000):
        V = unit(rng.normal(size=(n, 64)))
        w = softmax(rng.normal(size=n) * temp)
        ys.append(np.linalg.norm(w @ V))
        mw.append(w.max())
        ent.append(-(w * np.log(w + 1e-30)).sum())
        pred.append(np.sqrt((w ** 2).sum()))
    print(f"{n:>4}{name:>12}{np.mean(mw):>9.3f}{np.mean(ent):>10.3f}"
          f"{np.mean(ys):>9.3f}{np.mean(pred):>16.3f}")

The last two columns agree to three decimals everywhere, which is the useful part.

Note what `sum(w**2)` is: the same quantity lesson 074 measured as gradient sensitivity
`1 - sum(p**2)`. **So both failures of 074 are visible here as lengths.** Peaked attention
keeps the vector long and starves the gradient; flat attention keeps the gradient and
erases the vector down to 0.18 of what went in. Stack layers doing the second thing and
there is nothing left to pass on — which is the concrete reason a transformer block adds
the input back on (the residual connection) and renormalises the length (LayerNorm).

## Part D — The part of the picture that is wrong

The diagram shows `y_bank` ending up near `e_money`, among the original embeddings. That
only works because it quietly takes `W_v = I`. Make `W_v` a real matrix and the values
live somewhere else entirely.

So: with a learned `W_v`, does "bank ends up nearer money" survive? And — the cell that
matters — **does a random vector of the same length do just as well?**

In [ ]:
d, hits, control = 16, 0, 0
worst_err = 0.0
for _ in range(20_000):
    e_m, e_b = rng.normal(size=d), rng.normal(size=d)
    W_v = rng.normal(size=(d, d)) / np.sqrt(d)
    v_m, v_b = e_m @ W_v, e_b @ W_v
    y = 0.2 * v_m + 0.8 * v_b

    hits += np.linalg.norm(y - e_m) < np.linalg.norm(e_b - e_m)

    z = rng.normal(size=d)                      # knows nothing about "money"
    z *= np.linalg.norm(y) / np.linalg.norm(z)  # same length as y
    control += np.linalg.norm(z - e_m) < np.linalg.norm(e_b - e_m)

    frac = 1 - np.linalg.norm(y - v_m) / np.linalg.norm(v_b - v_m)
    worst_err = max(worst_err, abs(frac - 0.2))

print(f"y_bank nearer e_money       : {hits / 200:.1f}%")
print(f"RANDOM vector, same length  : {control / 200:.1f}%")
print(f"in VALUE space, 20% of the way from v_bank to v_money — max error {worst_err:.1e}")

A vector that has never heard of "money" scores the same 68%.

**So the apparent attraction in embedding space is not attraction.** It is the shrink from
Part C: anything shorter lands nearer the middle of everything, and `e_money` is in the
middle of everything. The gravity picture is not wrong — it is *drawn in the wrong space*.
In value space the same statement is exact to 1e-16, for every `W_v`, at every weight.

Say "the **value** of bank moves toward the **value** of money" and every word of it is
measurable.

## Part E — On real embeddings

Enough hypothetical 2×2 matrices. Build the 60-dimensional PPMI + SVD embeddings from
lesson 073, take the polysemous word *drive*, and watch where it goes in two sentences.

(The first run downloads the 20 newsgroups corpus, about 14 MB.)

In [ ]:
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import CountVectorizer

docs = fetch_20newsgroups(subset="train", remove=("headers", "footers", "quotes")).data
counts = CountVectorizer(max_features=4000, stop_words="english", min_df=5)
Xc = counts.fit_transform(docs)
vocab = counts.get_feature_names_out()
index = {w: i for i, w in enumerate(vocab)}

binary = (Xc > 0).astype(np.float32)
co = (binary.T @ binary).toarray()
np.fill_diagonal(co, 0)
tot = co.sum()
marg = co.sum(1) / tot
with np.errstate(divide="ignore", invalid="ignore"):
    ppmi = np.maximum(0, np.log((co / tot) / np.outer(marg, marg) + 1e-12))
U, S, _ = np.linalg.svd(ppmi, full_matrices=False)
E = unit(U[:, :60] * S[:60])
print(E.shape, "unit norm:", np.allclose(np.linalg.norm(E, axis=1), 1))

In [ ]:
SENTS = [["disk", "drive", "controller", "memory"],
         ["car", "drive", "engine", "road"]]
out = {}
for sent in SENTS:
    X = E[[index[w] for w in sent]]
    W = softmax(X @ X.T / np.sqrt(X.shape[1]))       # parameterless self-attention
    Y = W @ X
    b = sent.index("drive")
    out[sent[0]] = (sent, X, Y)
    print(f"\n\"{' '.join(sent)}\"   weights on 'drive': {np.round(W[b], 3)}")
    for j, w in enumerate(sent):
        if j != b:
            print(f"    cos(y_drive, e_{w:<11}) {cos(X[b], X[j]):.4f} -> {cos(Y[b], X[j]):.4f}")

ya = out["disk"][2][SENTS[0].index("drive")]
yb = out["car"][2][SENTS[1].index("drive")]
print(f"\ncos(y_drive disk-sense, y_drive car-sense) = {cos(ya, yb):.4f}")
print("a static embedding would report 1.0000 here, by definition")

Every cosine rose — that is the pull, on real data, with no training at all. And the two
outputs are genuinely different vectors, which is the whole purpose of the mechanism:
one word, two representations, chosen by context.

Now the bill. Look at what happened to the *other* words.

In [ ]:
for key, (sent, X, Y) in out.items():
    n = len(sent)
    pair = lambda M: np.mean([cos(M[i], M[j]) for i in range(n) for j in range(i + 1, n)])
    print(f"\"{' '.join(sent)}\"  mean pairwise cosine {pair(X):.4f} in -> {pair(Y):.4f} out")

**Everything landed on the same point.** Attention weights of 0.249, 0.253, 0.251, 0.248
are not a mechanism choosing anything — they are an average, and an average of everything
is the same vector no matter which word asked for it.

Geometrically it is obvious in hindsight: *gravity with no parameters has exactly one
attractor, the barycentre, and one layer is enough to reach it.* That is lesson 073's
collapse, seen as a picture, and it is why `W_q` and `W_k` exist — so that different words
fall toward different places.

## Try it yourself

1. Set `W_v = I` in Part D and confirm the 68% jumps to 100% — then explain why that is
   the *less* honest setup, not the more honest one.
2. In Part C, find the attention distribution over 30 words that gives `||y|| = 0.5`. How
   peaked does it have to be?
3. Take a three-word sentence in 2-D, plot the triangle of its embeddings, and scatter the
   outputs for 500 random weightings. Every point should be inside.
4. Replace the parameterless scores in Part E with `X @ Wq @ (X @ Wk).T` for random `Wq`,
   `Wk` and re-measure the pairwise cosine. Does the collapse go away? (Lesson 073 measured
   this — untrained projections collapse it *more*. Confirm it, then work out why.)